In [1]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit
import awkward as ak
import pandas as pd
import uproot
import os

In [3]:
def read_file(filename):
    file = uproot.open(filename)["pulse"].arrays()
    return file

def gaussian(x, A, mean, std):
    return A * np.exp(-1*(1/2)*((x-mean)/std)**2)

def timewalk_correction(dt, amp):
    h, xedges, yedges, image = plt.hist2d(amp, dt)
    plt.show()

def timing_resolution(filename):
    data = read_file(f"{path}/{filename}")
    os.system(f"mkdir -p plots")
    os.system(f"mkdir ./plots/{filename}/")
    # output_file = open(f"./plots/{filename}/output_params.csv", "w")
    signal_cut_efficiency = []
    timing_resolution = []
    chi_squared = []
    # branches = data[0].fields
    branches = ['LP2_5', 'LP2_10', 'LP2_15', 'LP2_20', 'LP2_25', 'LP2_30', 'LP2_35', 'LP2_40', 'LP2_50', 'LP2_60', 'LP2_70', 'LP2_80']
    quadratic_fit = [5, 10, 15, 20, 25, 30, 35, 40, 50, 60, 70, 80]
    print(branches)
    trigger = 0
    signal = 3
    n_bins = 60
    n_rows = 2
    for b, branch in enumerate(branches):
        fig, ax = plt.subplots(1, n_rows, figsize = (10*n_rows, 7))
        cut = (data[branch][:, trigger] < 95) # & (data[branch][:, signal] < data[branch][:, signal] + 10)
        signal_cut_efficiency.append(np.sum(cut)/len(cut))
        ax[0].hist2d(np.array(data[branch][:, signal][cut]), np.array(data[branch][:, trigger][cut]), bins=n_bins, norm=LogNorm())
        ax[0].set_ylabel("Trigger TOA [ns]")
        ax[0].set_xlabel("Signal TOA [ns]")
        ax[0].set_title(branch)
        deltaT = np.array(data[branch][:, signal][cut])-np.array(data[branch][:, trigger][cut])
        mean = np.mean(deltaT)
        std = np.std(deltaT)
        std_spacing = 2
        histo_num = ax[1].hist(deltaT, bins=n_bins, histtype="step", range=(mean-std_spacing*std, mean+std_spacing*std), linewidth=2)
        x_axis = histo_num[1][0:-1]
        y_axis = histo_num[0]
        p0 = (mean, mean, std)
        popt, pcov = curve_fit(gaussian, x_axis, y_axis, p0=p0)
        fit_amp = np.round(popt[0],2)
        fit_mean = np.round(popt[1],2)
        fit_sigma = np.abs(np.round(popt[2]*1000,2))
        print(p0)
        print(popt)
        timing_resolution.append(fit_sigma)
        chi_sq = ((y_axis - gaussian(x_axis, *popt))**2)/y_axis
        print(f"Difference {quadratic_fit[b]}: ", (y_axis - gaussian(x_axis, *popt))**2)
        print(f"Ratio {quadratic_fit[b]}: ", chi_sq)
        chi_squared.append(np.sum(chi_sq[np.isfinite(chi_sq)])/n_bins)
        print(f"Chi squared {quadratic_fit[b]}:", np.sum(chi_sq[np.isfinite(chi_sq)]))
        ax[1].plot(x_axis, gaussian(x_axis, *popt), 'r-', label=f'Fit: A={fit_amp}, mean={fit_mean} ns, sigma={fit_sigma} ps')
        ax[1].set_ylabel("Events")
        ax[1].set_xlabel("ΔT [ns]")
        ax[1].set_title(branch)
        ax[1].legend()
        fig.savefig(f"./plots/{filename}/{branch}.png")
    print("chi squared", chi_squared)
    fig, ax = plt.subplots(1, 2, figsize = (20, 7))
    ax[0].set_ylabel("Resolution [ps]")
    ax[0].set_xlabel("Fit amplitude %")
    ax[0].plot(quadratic_fit, timing_resolution, linewidth=2, marker="o", label = f"Best resolution: {min(timing_resolution)} ps")
    ax[0].legend()
    ax[1].set_ylabel("Chi^2/nof")
    ax[1].set_xlabel("Fit amplitude %")
    ax[1].plot(quadratic_fit, chi_squared, linewidth=2, marker="o", label = f"Best fit accuracy, χ²: {min(chi_squared)}")
    ax[1].legend()
    df = pd.DataFrame({"signal_cut_efficiency":signal_cut_efficiency, "timing_resolution":timing_resolution})
    os.system(f"rm ./plots/{filename}/output_params.csv")
    df.to_csv(f"./plots/{filename}/output_params.csv")
    fig.savefig(f"./plots/{filename}/timing_resolution.png")


In [6]:
path = "/home/aram/3D_sensors"
filename = "proc_F_AC100W80_strip3mm_bv116_mcp4400_trigCh4_Ch8_collimatedTwice_3days.root"
file_path = f"{path}/{filename}"
data = read_file(file_path)
branches = data[0].fields
print(branches)

['i_evt', 'channel', 'time', 'baseline', 'baseline_RMS', 'noise', 'amp', 't_peak', 'integral', 'intfull', 'risetime', 'decaytime', 'LP2_5', 'LP2_10', 'LP2_15', 'LP2_20', 'LP2_25', 'LP2_30', 'LP2_35', 'LP2_40', 'LP2_45', 'LP2_50', 'LP2_55', 'LP2_60', 'LP2_65', 'LP2_70', 'LP2_75', 'LP2_80', 'LP2_85', 'LP2_90', 'LP2_95', 'LP2_100', 'LP2_10mV', 'LP2_20mV', 'LP2_30mV', 'LP2_50mV', 'LP2_70mV', 'LP2_90mV', 'LP2_100mV', 'LP2_120mV', 'LP2_140mV', 'LP2_160mV', 'LP2_180mV', 'LP2_200mV', 'linear_RE_5', 'linear_RE_10', 'linear_RE_15', 'linear_RE_20', 'linear_RE_25', 'linear_RE_30', 'linear_RE_35', 'linear_RE_40', 'linear_RE_45', 'linear_RE_50', 'linear_RE_55', 'linear_RE_60', 'linear_RE_65', 'linear_RE_70', 'linear_RE_75', 'linear_RE_80', 'linear_RE_85', 'linear_RE_90', 'linear_RE_95', 'linear_RE_100', 'linear_RE__10mV', 'linear_RE__20mV', 'linear_RE__30mV', 'linear_RE__50mV', 'linear_RE__70mV', 'linear_RE__90mV', 'linear_RE__100mV', 'linear_RE__120mV', 'linear_RE__140mV', 'linear_RE__160mV', 'line

In [ ]:
T_CH4 = data["LP2_20"][:, 3]
T_MCP = data["LP2_20"][:, 7]
A_CH4 = data["LP2_20"][:, 3]
A_MCP = data["LP2_20"][:, 7]
DeltaT = T_MCP-T_CH4